In [34]:
from pyomo.environ import *
import pandas as pd
from pathlib import Path

BASE = Path.cwd()

DA_CSV_PATH = BASE / "DA_LMPs_Data/MISO/df_DA_MISO_MINN.HUB_2025_DA_hourly.csv"
RT_CSV_PATH = BASE / "RT_LMPs_Data/MISO/df_RT_MISO_MINN.HUB_2025_RT_5min.csv"
#REG_CSV_PATH = BASE /"reg_zone_prelim_bill_PJM_January_2025.csv"
#SIGNALS_CSV_PATH = BASE /"AGC Signal for Regulation/PJM/01_2025_avg_5min_pos_neg.csv"
EV_CSV_PATH = BASE / "EV_fleet_12h_night_charging.csv"

# Time Window used
TZ = "America/New_York"
pt_start = pd.Timestamp("2025-09-02 14:00", tz=TZ)
pt_end   = pd.Timestamp("2025-09-03 14:00", tz=TZ)

# Convert to UTC
utc_start = pt_start.tz_convert("UTC")
utc_end   = pt_end.tz_convert("UTC")

# Load Datasets
df_DA_all = pd.read_csv(DA_CSV_PATH)
df_RT_all = pd.read_csv(RT_CSV_PATH)
#df_REG_all = pd.read_csv(REG_CSV_PATH)
df_EV = pd.read_csv(EV_CSV_PATH)

# Parse the timestamp columns
df_DA_all["interval_start_utc"] = pd.to_datetime(df_DA_all["interval_start_utc"], utc=True)
df_RT_all["interval_start_utc"] = pd.to_datetime(df_RT_all["interval_start_utc"], utc=True)

df_DA = df_DA_all.loc[
    (df_DA_all["interval_start_utc"] >= utc_start) &
    (df_DA_all["interval_start_utc"] < utc_end)
].sort_values("interval_start_utc").reset_index(drop=True)

# =========================================================
# RT FILTER + INTERPOLATE MISSING 5-MIN PRICES
# =========================================================
df_RT_day = df_RT_all.loc[
    (df_RT_all["interval_start_utc"] >= utc_start) &
    (df_RT_all["interval_start_utc"] < utc_end)
].sort_values("interval_start_utc").copy()

# Full expected 5-minute index for the selected day
full_rt_index = pd.date_range(
    start=utc_start,
    end=utc_end - pd.Timedelta(minutes=5),
    freq="5min",
    tz="UTC"
)

# Reindex to insert missing timestamps
df_RT = (
    df_RT_day
    .drop_duplicates(subset="interval_start_utc")
    .set_index("interval_start_utc")
    .reindex(full_rt_index)
)

# Interpolate missing prices using previous and next available values
df_RT["lmp"] = df_RT["lmp"].interpolate(method="time")

# In case missing values are at the edges
df_RT["lmp"] = df_RT["lmp"].bfill().ffill()

# Back to normal dataframe
df_RT = df_RT.reset_index().rename(columns={"index": "interval_start_utc"})

# Optional checks
print(f"DA rows found: {len(df_DA)}")
print(f"RT rows after interpolation: {len(df_RT)}")
print(f"Remaining RT NaNs: {df_RT['lmp'].isna().sum()}")

#df_REG_all["datetime_beginning_ept"] = pd.to_datetime(df_REG_all["datetime_beginning_ept"])
#df_REG_all["datetime_beginning_ept"] = df_REG_all["datetime_beginning_ept"].dt.tz_localize(TZ)

#df_REG = df_REG_all.loc[
#    (df_REG_all["datetime_beginning_ept"] >= pt_start) &
#    (df_REG_all["datetime_beginning_ept"] <  pt_end)
#].sort_values("datetime_beginning_ept").reset_index(drop=True)

EV_ids = df_EV["EV"].tolist()

E_bat_max_dict = df_EV.set_index("EV")["E_bat_max (kWh)"].to_dict()
n_dict         = df_EV.set_index("EV")["n"].to_dict()
Pchrg_dict     = df_EV.set_index("EV")["Pchrg (kW)"].to_dict()
SOEa_dict      = df_EV.set_index("EV")["SOEa"].to_dict()
SOEd_dict      = df_EV.set_index("EV")["SOEd"].to_dict()

Ta_hr = df_EV.set_index("EV")["Ta_hr"].to_dict()
Td_hr = df_EV.set_index("EV")["Td_hr"].to_dict()

#T = len(df_DA)
#NK = len(df_RT)
T = 24                  # hours
K = 12                  # 5-minute steps per hour
NK = T * K
t0 = 14                 # initial hour of day
k0 = t0 * K             # initial 5-minute interval of day

# Align price dictionaries to absolute index sets
t_keys = list(range(t0, t0 + T))          # 14..37
k_keys = list(range(k0, k0 + NK))         # 168..455

# Safety checks
if len(df_DA) != T:
    raise ValueError(f"Expected {T} DA rows, found {len(df_DA)}")

if len(df_RT) != NK:
    raise ValueError(f"Expected {NK} RT rows after interpolation, found {len(df_RT)}")

# 1) DA prices: map 0-based rows -> absolute hours 14..37
lmp_DA = {t_keys[i]: float(df_DA.iloc[i]["lmp"]) for i in range(T)}

# 2) RT 5-min prices: map 0-based rows -> absolute sub-steps 168..455
lmp_RT = {k_keys[i]: float(df_RT.iloc[i]["lmp"]) for i in range(NK)}

# 3) Hourly RT averages over contiguous 12×5-min windows
lt_RTE = {}
for i, t in enumerate(t_keys):
    s = i * K
    e = (i + 1) * K
    lt_RTE[t] = float(df_RT.iloc[s:e]["lmp"].mean())

# Regulation RT clearing price
#lt_RMCCP = {t_keys[i]: float(df_REG.loc[i, "rmccp"]) for i in range(T)} #capacity
#lt_RMPCP = {t_keys[i]: float(df_REG.loc[i, "rmpcp"]) for i in range(T)} #mileage

# RT Signals
#RT_Signals = pd.read_csv(SIGNALS_CSV_PATH)
#cols = ["1/15/2025_pos", "1/15/2025_neg"]
#RT_Signals[cols] = RT_Signals[cols].fillna(0.0)

#r_pos_dict = {k_keys[i]: float(RT_Signals.loc[i, "1/15/2025_pos"]) for i in range(NK)}
#r_neg_dict = {k_keys[i]: float(RT_Signals.loc[i, "1/15/2025_neg"]) for i in range(NK)}

#def r_dc_pos_init(m, k, w):
#    return r_pos_dict[k]

#def r_dc_neg_init(m, k, w):
#    return -r_neg_dict[k]

#def hour_from_k(k):
#    return t0 + (k - k0) // K

## Build model
model = ConcreteModel()

## Sets
model.IDX_i = Set(initialize=EV_ids)
model.IDX_t = RangeSet(t0, t0 + T - 1)
model.IDX_k = RangeSet(k0, k0 + NK - 1)
model.IDX_w = Set(initialize=['w1'])             # single scenario

## Parameters
model.Dt     = Param(initialize=1/12)  # 5 minutes in hours
model.prob_w = Param(initialize=1.0)

# LMP Prices
model.l_DAE  = Param(model.IDX_t, initialize=lmp_DA)   # $/MWh
model.lk_RTE = Param(model.IDX_k, initialize=lmp_RT)   # $/MWh
model.lt_RTE = Param(model.IDX_t, initialize=lt_RTE)   # $/MWh

# Regulation Prices
#model.lt_RMCCP = Param(model.IDX_t, initialize=lt_RMCCP)  # capacity price
#model.lt_RMMCP = Param(model.IDX_t, initialize=lt_RMPCP)  # mileage price

# Reg Parameters
#model.s_perf = Param(model.IDX_t, initialize=0.985)
#model.a_mil = Param(model.IDX_t, initialize=1)

# EV fleet parameters
model.E_bat_max = Param(model.IDX_i, initialize=E_bat_max_dict)
model.n         = Param(model.IDX_i, initialize=n_dict)
model.Pchrg     = Param(model.IDX_i, initialize=Pchrg_dict)
model.SOEa      = Param(model.IDX_i, initialize=SOEa_dict)
model.SOEd      = Param(model.IDX_i, initialize=SOEd_dict)
model.SOEcc_cv  = Param(initialize=0.85)

# Regulation deployment ratios r
#model.r_pos = Param(model.IDX_k, model.IDX_w, initialize=r_dc_pos_init)
#model.r_neg = Param(model.IDX_k, model.IDX_w, initialize=r_dc_neg_init)

# Availability windows (hours counted from 14:00 of the first day;)
Ta_k = {i: int(Ta_hr[i]) * K for i in EV_ids}
Td_k = {i: int(Td_hr[i]) * K for i in EV_ids}

def u_init(model, i, k):
    return 1 if (k >= Ta_k[i] and k < Td_k[i]) else 0

model.u = Param(model.IDX_i, model.IDX_k, initialize=u_init, within=Binary)

# DART Spread Penalty settings
dev = 0.3
max_gap = max(0.0, max(lt_RTE[t] - lmp_DA[t] for t in t_keys))
M_fixed = 1.01 / (1.0 - dev) * max_gap
model.M = Param(initialize=M_fixed)   # $/MWh

## Decision variables
model.E_DA       = Var(model.IDX_t, within=NonNegativeReals)                              # MWh
model.E_RT       = Var(model.IDX_k, model.IDX_w, within=NonNegativeReals)                 # MWh (agg over EVs)
model.E_i_RT     = Var(model.IDX_i, model.IDX_k, model.IDX_w, within=NonNegativeReals)    # kWh
model.SOE        = Var(model.IDX_i, model.IDX_k, model.IDX_w, bounds=(0, 1))              # fraction
model.P_RT_max   = Var(model.IDX_i, model.IDX_k, model.IDX_w, within=NonNegativeReals)    # kW
model.DE         = Var(model.IDX_t, model.IDX_w)                                           # signed deviation (MWh)
model.DE_U       = Var(model.IDX_t, model.IDX_w)
#model.DE_I      = Var(model.IDX_t, model.IDX_w)
model.DE_U_Up    = Var(model.IDX_t, model.IDX_w, within=NonNegativeReals)
model.DE_U_Down  = Var(model.IDX_t, model.IDX_w, within=NonNegativeReals)
model.Pnlty_UUp   = Var(model.IDX_t, model.IDX_w, within=NonNegativeReals)
model.Pnlty_UDown = Var(model.IDX_t, model.IDX_w, within=NonNegativeReals)
model.b          = Var(model.IDX_t, model.IDX_w, within=Binary)

## Constraints
def DE_constraint(model, t, w):
    k0_t, k1_t = t * K, (t + 1) * K
    return model.DE[t, w] == model.E_DA[t] - sum(model.E_RT[k, w] for k in range(k0_t, k1_t))
model.DEcon = Constraint(model.IDX_t, model.IDX_w, rule=DE_constraint)

model.DE2con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.DE[t, w] == m.DE_U[t, w])
model.DE3con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.DE_U[t, w] == m.DE_U_Up[t, w] - m.DE_U_Down[t, w])

def de_up_limit(m, t, w):
    return m.DE_U_Up[t, w] <= 10**6 * m.b[t, w]
model.de_up_limit = Constraint(model.IDX_t, model.IDX_w, rule=de_up_limit)

def de_dn_limit(m, t, w):
    return m.DE_U_Down[t, w] <= 10**6 * (1 - m.b[t, w])
model.de_dn_limit = Constraint(model.IDX_t, model.IDX_w, rule=de_dn_limit)

def de_pos_enforce(m, t, w):
    return m.DE[t, w] >= -10**6 * (1 - m.b[t, w])
model.de_pos_enforce = Constraint(model.IDX_t, model.IDX_w, rule=de_pos_enforce)

def de_neg_enforce(m, t, w):
    return m.DE[t, w] <= 10**6 * m.b[t, w]
model.de_neg_enforce = Constraint(model.IDX_t, model.IDX_w, rule=de_neg_enforce)

model.EVcon = Constraint(
    model.IDX_k, model.IDX_w,
    rule=lambda m, k, w: m.E_RT[k, w] == sum(m.E_i_RT[i, k, w] for i in m.IDX_i) / 1000.0
)

model.EV2con = Constraint(
    model.IDX_i, model.IDX_k, model.IDX_w,
    rule=lambda m, i, k, w: m.E_i_RT[i, k, w] <= m.P_RT_max[i, k, w] * m.Dt
)

model.EV3con = Constraint(
    model.IDX_i, model.IDX_k, model.IDX_w,
    rule=lambda m, i, k, w: m.P_RT_max[i, k, w] <= m.u[i, k] * m.Pchrg[i]
)

def EV4_rule(model, i, k, w):
    return model.P_RT_max[i, k, w] <= model.u[i, k] * model.Pchrg[i] * ((1 - model.SOE[i, k, w]) / (1 - model.SOEcc_cv))
model.EV4con = Constraint(model.IDX_i, model.IDX_k, model.IDX_w, rule=EV4_rule)

def SOE_init_k0(model, i, w):
    return model.SOE[i, k0, w] == model.SOEa[i]
model.SOE_init_k0 = Constraint(model.IDX_i, model.IDX_w, rule=SOE_init_k0)

def EV5_rule(model, i, k, w):
    if k == k0 + NK - 1:
        return Constraint.Skip
    return model.SOE[i, k + 1, w] == model.SOE[i, k, w] + (model.n[i] / model.E_bat_max[i]) * model.E_i_RT[i, k, w]
model.EV5con = Constraint(model.IDX_i, model.IDX_k, model.IDX_w, rule=EV5_rule)

def EV6_rule(model, i, w):
    k_dep = Td_k[i] - 1
    if k0 <= k_dep < k0 + NK:
        return model.SOE[i, k_dep, w] == model.SOEd[i]
    return Constraint.Skip
model.EV6con = Constraint(model.IDX_i, model.IDX_w, rule=EV6_rule)

model.Pcon  = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.Pnlty_UUp[t, w] >= 0)
model.P3con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.Pnlty_UDown[t, w] >= 0)
model.P2con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.Pnlty_UUp[t, w] >= m.M * (m.DE_U_Up[t, w] - dev * m.E_DA[t]))
model.P4con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.Pnlty_UDown[t, w] >= m.M * (m.DE_U_Down[t, w] - dev * m.E_DA[t]))

## Objective Function
def objFunc_(model):
    w = 'w1'
    return sum(
        -model.l_DAE[t] * model.E_DA[t]
        + model.lt_RTE[t] * model.DE[t, w]
        - model.Pnlty_UUp[t, w] - model.Pnlty_UDown[t, w]
        for t in model.IDX_t
    )

model.cost = Objective(rule=objFunc_, sense=maximize)

# solve
solver = SolverFactory("gurobi")

# GUROBI options
solver.options["MIPGap"] = 1e-4
solver.options["TimeLimit"] = 300  # seconds

results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
print(f"Total Profit: ${value(model.cost):.2f}")

# Results printed
da_series = pd.Series({t: value(model.E_DA[t]) for t in model.IDX_t}, name="E_DA_MWh")
de_series = pd.Series({t: value(model.DE[t, 'w1']) for t in model.IDX_t}, name="DE_MWh")
prices_da = pd.Series({t: value(model.l_DAE[t]) for t in model.IDX_t}, name="DA_$perMWh")
prices_rt = pd.Series({t: value(model.lt_RTE[t]) for t in model.IDX_t}, name="RTavg_$perMWh")

out = pd.concat(
    [prices_da, prices_rt, da_series, de_series],
    axis=1
)

print(out.head(24).round(3))

OUTPUT_PATH = BASE / "EV_aggregator_energy_sched.csv"
out.to_csv(OUTPUT_PATH, index=True)

print(f"\nResults saved to:\n{OUTPUT_PATH}")

DA rows found: 24
RT rows after interpolation: 288
Remaining RT NaNs: 0
Read LP format model from file C:\Users\hsofi\AppData\Local\Temp\tmpoujaph8v.pyomo.lp
Reading time = 0.97 seconds
x1: 230952 rows, 173280 columns, 480728 nonzeros
Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 165U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 230952 rows, 173280 columns and 480728 nonzeros (Max)
Model fingerprint: 0x66e5fe8c
Model has 96 linear objective coefficients
Variable types: 173256 continuous, 24 integer (24 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+06]
  Objective range  [1e+00, 9e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 1e+06]

Presolve removed 218536 rows and 150032 colu

In [54]:
#Day_Charging
from pyomo.environ import *
import pandas as pd
from pathlib import Path

BASE = Path.cwd()

DA_CSV_PATH = BASE / "DA_LMPs_Data/MISO/DA_LMPs_SOUTH_MISO_AVGZONES.csv"
RT_CSV_PATH = BASE / "RT_LMPs_Data/MISO/RT_LMPs_SOUTH_MISO_AVGZONES.csv"
#REG_CSV_PATH = BASE /"reg_zone_prelim_bill_PJM_January_2025.csv"
#SIGNALS_CSV_PATH = BASE /"AGC Signal for Regulation/PJM/01_2025_avg_5min_pos_neg.csv"
EV_CSV_PATH = BASE / "EV_fleet_12h_day_charging.csv"

# Time Window used
TZ = "America/New_York"
pt_start = pd.Timestamp("2025-10-09 14:00", tz=TZ)
pt_end   = pd.Timestamp("2025-10-10 14:00", tz=TZ)

# Convert to UTC
utc_start = pt_start.tz_convert("UTC")
utc_end   = pt_end.tz_convert("UTC")

# Load Datasets
df_DA_all = pd.read_csv(DA_CSV_PATH)
df_RT_all = pd.read_csv(RT_CSV_PATH)
#df_REG_all = pd.read_csv(REG_CSV_PATH)
df_EV = pd.read_csv(EV_CSV_PATH)

# Parse the timestamp columns
df_DA_all["interval_start_utc"] = pd.to_datetime(df_DA_all["interval_start_utc"], utc=True)
df_RT_all["interval_start_utc"] = pd.to_datetime(df_RT_all["interval_start_utc"], utc=True)

df_DA = df_DA_all.loc[
    (df_DA_all["interval_start_utc"] >= utc_start) &
    (df_DA_all["interval_start_utc"] < utc_end)
].sort_values("interval_start_utc").reset_index(drop=True)

# =========================================================
# RT FILTER + INTERPOLATE MISSING 5-MIN PRICES
# =========================================================
df_RT_day = df_RT_all.loc[
    (df_RT_all["interval_start_utc"] >= utc_start) &
    (df_RT_all["interval_start_utc"] < utc_end)
].sort_values("interval_start_utc").copy()

# Full expected 5-minute index for the selected day
full_rt_index = pd.date_range(
    start=utc_start,
    end=utc_end - pd.Timedelta(minutes=5),
    freq="5min",
    tz="UTC"
)

# Reindex to insert missing timestamps
df_RT = (
    df_RT_day
    .drop_duplicates(subset="interval_start_utc")
    .set_index("interval_start_utc")
    .reindex(full_rt_index)
)

# Interpolate missing prices using previous and next available values
df_RT["lmp"] = df_RT["lmp"].interpolate(method="time")

# In case missing values are at the edges
df_RT["lmp"] = df_RT["lmp"].bfill().ffill()

# Back to normal dataframe
df_RT = df_RT.reset_index().rename(columns={"index": "interval_start_utc"})

# Optional checks
print(f"DA rows found: {len(df_DA)}")
print(f"RT rows after interpolation: {len(df_RT)}")
print(f"Remaining RT NaNs: {df_RT['lmp'].isna().sum()}")

#df_REG_all["datetime_beginning_ept"] = pd.to_datetime(df_REG_all["datetime_beginning_ept"])
#df_REG_all["datetime_beginning_ept"] = df_REG_all["datetime_beginning_ept"].dt.tz_localize(TZ)

#df_REG = df_REG_all.loc[
#    (df_REG_all["datetime_beginning_ept"] >= pt_start) &
#    (df_REG_all["datetime_beginning_ept"] <  pt_end)
#].sort_values("datetime_beginning_ept").reset_index(drop=True)

EV_ids = df_EV["EV"].tolist()

E_bat_max_dict = df_EV.set_index("EV")["E_bat_max (kWh)"].to_dict()
n_dict         = df_EV.set_index("EV")["n"].to_dict()
Pchrg_dict     = df_EV.set_index("EV")["Pchrg (kW)"].to_dict()
SOEa_dict      = df_EV.set_index("EV")["SOEa"].to_dict()
SOEd_dict      = df_EV.set_index("EV")["SOEd"].to_dict()

Ta_hr = df_EV.set_index("EV")["Ta_hr"].to_dict()
Td_hr = df_EV.set_index("EV")["Td_hr"].to_dict()

#T = len(df_DA)
#NK = len(df_RT)
T = 24                  # hours
K = 12                  # 5-minute steps per hour
NK = T * K
t0 = 18                 # initial hour of day
k0 = t0 * K             # initial 5-minute interval of day

# Align price dictionaries to absolute index sets
t_keys = list(range(t0, t0 + T))          # 14..37
k_keys = list(range(k0, k0 + NK))         # 168..455

# Safety checks
if len(df_DA) != T:
    raise ValueError(f"Expected {T} DA rows, found {len(df_DA)}")

if len(df_RT) != NK:
    raise ValueError(f"Expected {NK} RT rows after interpolation, found {len(df_RT)}")

# 1) DA prices: map 0-based rows -> absolute hours 14..37
lmp_DA = {t_keys[i]: float(df_DA.iloc[i]["lmp"]) for i in range(T)}

# 2) RT 5-min prices: map 0-based rows -> absolute sub-steps 168..455
lmp_RT = {k_keys[i]: float(df_RT.iloc[i]["lmp"]) for i in range(NK)}

# 3) Hourly RT averages over contiguous 12×5-min windows
lt_RTE = {}
for i, t in enumerate(t_keys):
    s = i * K
    e = (i + 1) * K
    lt_RTE[t] = float(df_RT.iloc[s:e]["lmp"].mean())

# Regulation RT clearing price
#lt_RMCCP = {t_keys[i]: float(df_REG.loc[i, "rmccp"]) for i in range(T)} #capacity
#lt_RMPCP = {t_keys[i]: float(df_REG.loc[i, "rmpcp"]) for i in range(T)} #mileage

# RT Signals
#RT_Signals = pd.read_csv(SIGNALS_CSV_PATH)
#cols = ["1/15/2025_pos", "1/15/2025_neg"]
#RT_Signals[cols] = RT_Signals[cols].fillna(0.0)

#r_pos_dict = {k_keys[i]: float(RT_Signals.loc[i, "1/15/2025_pos"]) for i in range(NK)}
#r_neg_dict = {k_keys[i]: float(RT_Signals.loc[i, "1/15/2025_neg"]) for i in range(NK)}

#def r_dc_pos_init(m, k, w):
#    return r_pos_dict[k]

#def r_dc_neg_init(m, k, w):
#    return -r_neg_dict[k]

#def hour_from_k(k):
#    return t0 + (k - k0) // K

## Build model
model = ConcreteModel()

## Sets
model.IDX_i = Set(initialize=EV_ids)
model.IDX_t = RangeSet(t0, t0 + T - 1)
model.IDX_k = RangeSet(k0, k0 + NK - 1)
model.IDX_w = Set(initialize=['w1'])             # single scenario

## Parameters
model.Dt     = Param(initialize=1/12)  # 5 minutes in hours
model.prob_w = Param(initialize=1.0)

# LMP Prices
model.l_DAE  = Param(model.IDX_t, initialize=lmp_DA)   # $/MWh
model.lk_RTE = Param(model.IDX_k, initialize=lmp_RT)   # $/MWh
model.lt_RTE = Param(model.IDX_t, initialize=lt_RTE)   # $/MWh

# Regulation Prices
#model.lt_RMCCP = Param(model.IDX_t, initialize=lt_RMCCP)  # capacity price
#model.lt_RMMCP = Param(model.IDX_t, initialize=lt_RMPCP)  # mileage price

# Reg Parameters
#model.s_perf = Param(model.IDX_t, initialize=0.985)
#model.a_mil = Param(model.IDX_t, initialize=1)

# EV fleet parameters
model.E_bat_max = Param(model.IDX_i, initialize=E_bat_max_dict)
model.n         = Param(model.IDX_i, initialize=n_dict)
model.Pchrg     = Param(model.IDX_i, initialize=Pchrg_dict)
model.SOEa      = Param(model.IDX_i, initialize=SOEa_dict)
model.SOEd      = Param(model.IDX_i, initialize=SOEd_dict)
model.SOEcc_cv  = Param(initialize=0.85)

# Regulation deployment ratios r
#model.r_pos = Param(model.IDX_k, model.IDX_w, initialize=r_dc_pos_init)
#model.r_neg = Param(model.IDX_k, model.IDX_w, initialize=r_dc_neg_init)

# Availability windows (hours counted from 14:00 of the first day;)
Ta_k = {i: int(Ta_hr[i]) * K for i in EV_ids}
Td_k = {i: int(Td_hr[i]) * K for i in EV_ids}

def u_init(model, i, k):
    return 1 if (k >= Ta_k[i] and k < Td_k[i]) else 0

model.u = Param(model.IDX_i, model.IDX_k, initialize=u_init, within=Binary)

# DART Spread Penalty settings
dev = 0.3
max_gap = max(0.0, max(lt_RTE[t] - lmp_DA[t] for t in t_keys))
M_fixed = 1.01 / (1.0 - dev) * max_gap
model.M = Param(initialize=M_fixed)   # $/MWh

## Decision variables
model.E_DA       = Var(model.IDX_t, within=NonNegativeReals)                              # MWh
model.E_RT       = Var(model.IDX_k, model.IDX_w, within=NonNegativeReals)                 # MWh (agg over EVs)
model.E_i_RT     = Var(model.IDX_i, model.IDX_k, model.IDX_w, within=NonNegativeReals)    # kWh
model.SOE        = Var(model.IDX_i, model.IDX_k, model.IDX_w, bounds=(0, 1))              # fraction
model.P_RT_max   = Var(model.IDX_i, model.IDX_k, model.IDX_w, within=NonNegativeReals)    # kW
model.DE         = Var(model.IDX_t, model.IDX_w)                                           # signed deviation (MWh)
model.DE_U       = Var(model.IDX_t, model.IDX_w)
#model.DE_I      = Var(model.IDX_t, model.IDX_w)
model.DE_U_Up    = Var(model.IDX_t, model.IDX_w, within=NonNegativeReals)
model.DE_U_Down  = Var(model.IDX_t, model.IDX_w, within=NonNegativeReals)
model.Pnlty_UUp   = Var(model.IDX_t, model.IDX_w, within=NonNegativeReals)
model.Pnlty_UDown = Var(model.IDX_t, model.IDX_w, within=NonNegativeReals)
model.b          = Var(model.IDX_t, model.IDX_w, within=Binary)

## Constraints
def DE_constraint(model, t, w):
    k0_t, k1_t = t * K, (t + 1) * K
    return model.DE[t, w] == model.E_DA[t] - sum(model.E_RT[k, w] for k in range(k0_t, k1_t))
model.DEcon = Constraint(model.IDX_t, model.IDX_w, rule=DE_constraint)

model.DE2con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.DE[t, w] == m.DE_U[t, w])
model.DE3con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.DE_U[t, w] == m.DE_U_Up[t, w] - m.DE_U_Down[t, w])

def de_up_limit(m, t, w):
    return m.DE_U_Up[t, w] <= 10**6 * m.b[t, w]
model.de_up_limit = Constraint(model.IDX_t, model.IDX_w, rule=de_up_limit)

def de_dn_limit(m, t, w):
    return m.DE_U_Down[t, w] <= 10**6 * (1 - m.b[t, w])
model.de_dn_limit = Constraint(model.IDX_t, model.IDX_w, rule=de_dn_limit)

def de_pos_enforce(m, t, w):
    return m.DE[t, w] >= -10**6 * (1 - m.b[t, w])
model.de_pos_enforce = Constraint(model.IDX_t, model.IDX_w, rule=de_pos_enforce)

def de_neg_enforce(m, t, w):
    return m.DE[t, w] <= 10**6 * m.b[t, w]
model.de_neg_enforce = Constraint(model.IDX_t, model.IDX_w, rule=de_neg_enforce)

model.EVcon = Constraint(
    model.IDX_k, model.IDX_w,
    rule=lambda m, k, w: m.E_RT[k, w] == sum(m.E_i_RT[i, k, w] for i in m.IDX_i) / 1000.0
)

model.EV2con = Constraint(
    model.IDX_i, model.IDX_k, model.IDX_w,
    rule=lambda m, i, k, w: m.E_i_RT[i, k, w] <= m.P_RT_max[i, k, w] * m.Dt
)

model.EV3con = Constraint(
    model.IDX_i, model.IDX_k, model.IDX_w,
    rule=lambda m, i, k, w: m.P_RT_max[i, k, w] <= m.u[i, k] * m.Pchrg[i]
)

def EV4_rule(model, i, k, w):
    return model.P_RT_max[i, k, w] <= model.u[i, k] * model.Pchrg[i] * ((1 - model.SOE[i, k, w]) / (1 - model.SOEcc_cv))
model.EV4con = Constraint(model.IDX_i, model.IDX_k, model.IDX_w, rule=EV4_rule)

def SOE_init_k0(model, i, w):
    return model.SOE[i, k0, w] == model.SOEa[i]
model.SOE_init_k0 = Constraint(model.IDX_i, model.IDX_w, rule=SOE_init_k0)

def EV5_rule(model, i, k, w):
    if k == k0 + NK - 1:
        return Constraint.Skip
    return model.SOE[i, k + 1, w] == model.SOE[i, k, w] + (model.n[i] / model.E_bat_max[i]) * model.E_i_RT[i, k, w]
model.EV5con = Constraint(model.IDX_i, model.IDX_k, model.IDX_w, rule=EV5_rule)

def EV6_rule(model, i, w):
    k_dep = Td_k[i] - 1
    if k0 <= k_dep < k0 + NK:
        return model.SOE[i, k_dep, w] == model.SOEd[i]
    return Constraint.Skip
model.EV6con = Constraint(model.IDX_i, model.IDX_w, rule=EV6_rule)

model.Pcon  = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.Pnlty_UUp[t, w] >= 0)
model.P3con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.Pnlty_UDown[t, w] >= 0)
model.P2con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.Pnlty_UUp[t, w] >= m.M * (m.DE_U_Up[t, w] - dev * m.E_DA[t]))
model.P4con = Constraint(model.IDX_t, model.IDX_w, rule=lambda m, t, w: m.Pnlty_UDown[t, w] >= m.M * (m.DE_U_Down[t, w] - dev * m.E_DA[t]))

## Objective Function
def objFunc_(model):
    w = 'w1'
    return sum(
        -model.l_DAE[t] * model.E_DA[t]
        + model.lt_RTE[t] * model.DE[t, w]
        - model.Pnlty_UUp[t, w] - model.Pnlty_UDown[t, w]
        for t in model.IDX_t
    )

model.cost = Objective(rule=objFunc_, sense=maximize)

# solve
solver = SolverFactory("gurobi")

# GUROBI options
solver.options["MIPGap"] = 1e-4
solver.options["TimeLimit"] = 300  # seconds

results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
print(f"Total Profit: ${value(model.cost):.2f}")

# Results printed
da_series = pd.Series({t: value(model.E_DA[t]) for t in model.IDX_t}, name="E_DA_MWh")
de_series = pd.Series({t: value(model.DE[t, 'w1']) for t in model.IDX_t}, name="DE_MWh")
prices_da = pd.Series({t: value(model.l_DAE[t]) for t in model.IDX_t}, name="DA_$perMWh")
prices_rt = pd.Series({t: value(model.lt_RTE[t]) for t in model.IDX_t}, name="RTavg_$perMWh")

out = pd.concat(
    [prices_da, prices_rt, da_series, de_series],
    axis=1
)

print(out.head(24).round(3))

OUTPUT_PATH = BASE / "EV_aggregator_energy_sched.csv"
out.to_csv(OUTPUT_PATH, index=True)

print(f"\nResults saved to:\n{OUTPUT_PATH}")

DA rows found: 24
RT rows after interpolation: 288
Remaining RT NaNs: 0
Read LP format model from file C:\Users\hsofi\AppData\Local\Temp\tmpg_uw91l1.pyomo.lp
Reading time = 0.91 seconds
x1: 231152 rows, 173280 columns, 490528 nonzeros
Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 165U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 231152 rows, 173280 columns and 490528 nonzeros (Max)
Model fingerprint: 0xe8115e71
Model has 96 linear objective coefficients
Variable types: 173256 continuous, 24 integer (24 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+06]
  Objective range  [1e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 1e+06]

Presolve removed 206508 rows and 138007 colu